In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4.1-mini'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/tencent/HY-MT1.5-1.8B',
 '/Qwen/Qwen-Image-2512',
 '/Lightricks/LTX-2',
 '/LGAI-EXAONE/K-EXAONE-236B-A23B',
 '/IQuestLab/IQuest-Coder-V1-40B-Loop-Instruct',
 '/models',
 '/spaces/Wan-AI/Wan2.2-Animate',
 '/spaces/mrfakename/Z-Image-Turbo',
 '/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast',
 '/spaces/Qwen/Qwen-Image-2512',
 '/spaces/selfit-camera/Omni-Image-Editor',
 '/spaces',
 '/datasets/facebook/research-plan-gen',
 '/datasets/Anthropic/hh-rlhf',
 '/datasets/genrobot2025/10Kh-RealOmin-OpenData',
 '/datasets/OpenDataArena/ODA-Mixture-500k',
 '/datasets/wikimedia/wikipedia',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/m

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://huggingface.co"))

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4.1-mini
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'company social media',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-4.1-mini
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
tencent/HY-MT1.5-1.8B
Updated
7 days ago
•
6.61k
•
652
Qwen/Qwen-Image-2512
Updated
8 days ago
•
16.8k
•
511
Lightricks/LTX-2
Updated
about 11 hours ago
•
84.4k
•
429
LGAI-EXAONE/K-EXAONE-236B-A23B
Updated
2 days ago
•
2.67k
•
421
IQuestLab/IQuest-Coder-V1-40B-Loop-Instruct
Updated
about 11 hours ago
•
6.32k
•
272
Browse 2M+ models
Spaces
Running
Featured
3.89k
Wan2.2 Animate
👁
3.89k
Wan2.2 Animate
Running
on
Zero
1.11k
Z Image Turbo
🖼
1.11k
Generate images from text prompts
Running
on
Zero
MCP
Featured
250
Qwen-Image-Edit-2511-LoRA

In [13]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    # response = ""
    # display_handle = display(Markdown(""), display_id=True)
    # for chunk in stream:
    #     response += chunk.choices[0].delta.content or ''
    #     update_display(Markdown(response), display_id=display_handle.display_id)
    response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta and delta.content:
            response += delta.content
            yield response

In [17]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4.1-mini
Found 5 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Welcome to Hugging Face: The AI Community Building the Future 🚀🤗

---

## Who Are We?

At Hugging Face, we’re not just another AI company — we’re **the** AI community where machine learning geeks, data wizards, and tech dreamers come to play, share, and build the future together. Founded in 2016 and proudly rooted in the City of Light (a.k.a. Paris), we’re a vibrant, fast-growing hub with **51-200 passionate employees** crafting the next generation of open and ethical AI.

---

## What’s Our Magic?

- **From Text to 3D**: Whether you want to decode language, generate photorealistic images, craft AI-powered apps, or even tinker with audio and video data, Hugging Face is your go-to playground.
- **2 Million+ Models & 500k+ Datasets**: Dive into our colossal AI treasure trove of models and datasets, all shared *publicly* by a buzzing community who love to collaborate.
- **Open Source Heroics**: We power the machine learning universe with some of the most loved open-source libraries and tools. Want to build your AI portfolio? Here’s your stage!
- **Spaces**: Host and share your AI demos, experiments or masterpieces with the world — because showing off is part of science.

---

## What’s It Like Working Here?

Think of us as the *Silicon Valley of AI*, mixed with a cozy Parisian café vibe:

- **Culture**: Friendly, open, and ethical with a golden heart—building AI that’s accessible to all.
- **Team & Enterprise Solutions**: We don’t just play with cool tech—we help teams accelerate with paid compute resources and enterprise-grade solutions.
- **Global Collaborations**: From Meta to PyTorch, we’re teaming up with the biggest brains and brands who share our vision.
- **Talent Welcome**: We’re always scouting for curious minds ready to dive into AI’s wild frontier. If you dream in Python, speak fluent ML, or just want to help build AI that feels almost human—check out our careers page!

---

## Our Customers & Community?

Whether you’re a:

- **Machine Learning Engineer** itching to experiment,
- **Scientist** chasing the bleeding edge,
- **Entrepreneur** dreaming of AI-powered products,
- or just an **enthusiast** fascinated by what machines can create—

Hugging Face is your tribe. With over 600 active collaborators and millions browsing and using our models weekly, we’re truly a global melting pot of ML creativity.

---

## Why Hugging Face? Because...

- You can **browse 2 million+ models** that someone else lovingly trained,
- Play with **spaces that host AI demos and apps** that range from image generators to code wizards,
- Ride the bleeding edge of **open, ethical AI development**,
- And join a team where your ideas won’t just sit in a drawer—they’ll live, breathe, and maybe even hug you back.

---

## Ready to Join the AI Revolution?

Explore, share, and create with us.

- **Start your journey**: [Explore AI Apps](https://huggingface.co/apps)
- **Browse the sprawling model zoo**: 2,000,000+ ready-to-run models
- **Build a career that hugs you back**: [Join Hugging Face](https://huggingface.co/careers)

---

## A Little Hug From Our Colors and Logo 🎨

Bright yellow (#FFD21E) and orange (#FF9D00) reflect our sunny optimism about AI's future, paired with cool charcoal (#6B7280) for that tech professional swagger. And yes, our logo? Think of a friendly smiling face wrapped in a warm, AI-powered embrace — because that’s how we roll.

---

# Hugging Face  
**Where AI meets Community, Creativity, and Care.**  
Come for the models, stay for the hugs. 🤗

---

*Disclaimer: No actual hugging required, but highly encouraged.*

In [ ]:
import gradio as gr

company_input = gr.Textbox(
    label="Company name",
    info="Name of the company",
    lines=1
)

url_input = gr.Textbox(
    label="Website URL",
    info="Give the URL of the company website",
    lines=1
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="GPT", 
    inputs=[company_input, url_input], 
    outputs=[message_output], 
    description="Generate a brochure for a company from its website using GPT-4.1-mini.",
    flagging_mode="never"
    )
view.launch() 

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://github.com/srinath-19 by calling gpt-4.1-mini
Found 5 relevant links
